# AAT6153 Chapter 1 — Hands-on 2
## Classifying MNIST Digits with a Multilayer Perceptron & Comparing Optimizers
**Slides:** Training Neural Networks (slides 13–19), MNIST dataset (slide 35), FC vs CNN parameters (slides 45–46)

In this notebook you will:
1. Explore the **MNIST** handwritten digit dataset.
2. Train a fully-connected network (**784-784-10**) with **softmax** output (slide 11).
3. Compare **SGD, SGD + momentum, SGD + learning-rate decay, RMSProp and Adam** (slides 14, 17, 18).
4. See why fully-connected networks have **so many parameters** for images, which motivates CNNs.

*Runtime: CPU works; GPU (Runtime ▸ Change runtime type ▸ T4 GPU) makes Part C faster.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import SGD, RMSprop, Adam
from tensorflow.keras.optimizers.schedules import ExponentialDecay

tf.keras.utils.set_random_seed(7)  # fix random seeds for reproducibility

---
# Part A — Explore MNIST (slide 35)
60,000 training images and 10,000 test images, each a 28 × 28 grayscale image of a digit 0–9.

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()
print("Training images:", X_train.shape, " labels:", y_train.shape)
print("Test images:    ", X_test.shape, " labels:", y_test.shape)
print("Pixel range:     ", X_train.min(), "to", X_train.max())

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(y_train[i]); ax.axis(False)
plt.tight_layout(); plt.show()

---
# Part B — Basic MLP for MNIST
A fully-connected network cannot take a 2-D image directly, so each 28 × 28 image is **flattened** into a 784-dimensional vector. Pixel values are scaled from 0–255 to 0–1, and labels are **one-hot encoded** for `categorical_crossentropy`.

In [ ]:
# flatten 28*28 images to a 784 vector for each image
num_pixels = X_train.shape[1] * X_train.shape[2]
X_train_flat = X_train.reshape(-1, num_pixels).astype('float32') / 255
X_test_flat = X_test.reshape(-1, num_pixels).astype('float32') / 255

# one hot encode outputs
y_train_oh = to_categorical(y_train)
y_test_oh = to_categorical(y_test)
num_classes = y_test_oh.shape[1]
print("Label", y_train[0], "→ one-hot", y_train_oh[0])

In [ ]:
def build_mlp():
    model = Sequential([Input(shape=(num_pixels,))])
    model.add(Dense(num_pixels, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))
    return model

model = build_mlp()
model.summary()

**Notice the parameter count.** The hidden layer alone has 784 × 784 + 784 = **615,440** parameters, for a tiny 28 × 28 image.
For a 250 × 250 image, *each* hidden neuron would need 62,501 weights (slide 45). This scalability problem is what convolutional layers solve (Hands-on 3 & 4).

In [ ]:
opt = SGD(learning_rate=0.01)
model.compile(loss='categorical_crossentropy', optimizer=opt, metrics=['accuracy'])
history = model.fit(X_train_flat, y_train_oh, validation_split=0.1,
                    epochs=20, batch_size=200, verbose=2)

scores = model.evaluate(X_test_flat, y_test_oh, verbose=0)
print(f"Test accuracy: {scores[1]:.2%}   Test error: {1 - scores[1]:.2%}")

In [ ]:
# Plot the accuracy and loss for both training and validation data
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'], 'b', label='train')
ax1.plot(history.history['val_accuracy'], 'g', label='val')
ax1.set(xlabel='Epoch', ylabel='Accuracy', title='Model Accuracy'); ax1.legend()
ax2.plot(history.history['loss'], 'b', label='train')
ax2.plot(history.history['val_loss'], 'g', label='val')
ax2.set(xlabel='Epoch', ylabel='Loss', title='Model Loss'); ax2.legend()
plt.show()

### Make a prediction
Change `index` to any number between 0 and 9999 and re-run the cell.

In [ ]:
index = 1234

res = model.predict(X_test_flat[index:index + 1], verbose=0)
print("Class probabilities:", res.round(3))
print(f"Predicted label: {res.argmax()}")

plt.axis(False)
plt.title(f'Actual label: {y_test[index]}')
plt.imshow(X_test[index], cmap='gray')
plt.show()

---
# Part C — Comparing optimizers (slides 14, 17, 18)
We train the **same** network from the **same** starting weights with five different optimizers for 10 epochs each and compare the validation accuracy curves.

| Optimizer | What it does |
|---|---|
| SGD | Plain gradient descent with a fixed learning rate |
| SGD + momentum | Keeps moving in the direction of previous updates (slide 18) |
| SGD + LR decay | Starts with a large learning rate and shrinks it over time (slide 18) |
| RMSProp | Adapts the learning rate per weight using recent gradient magnitudes |
| Adam | Combines momentum with RMSProp-style adaptive learning rates |

In [ ]:
steps_per_epoch = int(np.ceil(0.9 * len(X_train_flat) / 200))

optimizers = {
    'SGD':            lambda: SGD(learning_rate=0.01),
    'SGD + momentum': lambda: SGD(learning_rate=0.01, momentum=0.9),
    'SGD + LR decay': lambda: SGD(learning_rate=ExponentialDecay(
                                  initial_learning_rate=0.1,
                                  decay_steps=steps_per_epoch, decay_rate=0.7)),
    'RMSProp':        lambda: RMSprop(learning_rate=0.001),
    'Adam':           lambda: Adam(learning_rate=0.001),
}

tf.keras.utils.set_random_seed(7)
reference = build_mlp()
initial_weights = reference.get_weights()   # same starting point for every optimizer

results = {}
for name, make_opt in optimizers.items():
    m = build_mlp()
    m.set_weights(initial_weights)
    m.compile(loss='categorical_crossentropy', optimizer=make_opt(), metrics=['accuracy'])
    h = m.fit(X_train_flat, y_train_oh, validation_split=0.1,
              epochs=10, batch_size=200, verbose=0)
    test_acc = m.evaluate(X_test_flat, y_test_oh, verbose=0)[1]
    results[name] = (h.history, test_acc)
    print(f"{name:15s} test accuracy = {test_acc:.2%}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
for name, (hist, _) in results.items():
    epochs = np.arange(1, len(hist['val_accuracy']) + 1)
    ax1.plot(epochs, hist['val_accuracy'], marker='o', label=name)
    ax2.plot(epochs, hist['loss'], marker='o', label=name)
ax1.set(xlabel='Epoch', ylabel='Validation accuracy', title='Validation accuracy by optimizer')
ax2.set(xlabel='Epoch', ylabel='Training loss', title='Training loss by optimizer')
ax1.legend(); ax2.legend(); plt.show()

---
# Part D — Try it yourself
1. **Learning rate (slide 17):** re-run Part C with plain SGD at `learning_rate=0.1` and `0.001`. Which converges fastest? Which is unstable?
2. **Mini-batch size (slide 16):** change `batch_size` to 32 and 2000. How do training time and the curves change?
3. **Depth:** add a second hidden layer (e.g. 784-512-256-10). Does test accuracy improve?
4. **Early stopping (slide 15):** add `EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)` and train for 50 epochs.

In [ ]:
# Use this cell for your experiments